In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import butter, sosfiltfilt, welch

warnings.filterwarnings("ignore")

DATA_DIR = Path("../../data")
EPN_DIR = DATA_DIR / "epn612" / "raw"
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

FS = 200  # Hz, Myo armband sampling rate
GESTURES = ["noGesture", "open", "pinch", "waveOut", "fist", "waveIn"]
N_CHANNELS = 8
FOURIER_SEG_SECONDS = 2.0

DRIFT_CUTOFF_HZ = 20.0
_highpass_sos = butter(4, DRIFT_CUTOFF_HZ, btype="highpass", fs=FS, output="sos")


def apply_highpass(x: np.ndarray) -> np.ndarray:
    """Zero-phase Butterworth high-pass (removes DC offset and sub-20Hz drift)."""
    return sosfiltfilt(_highpass_sos, x, axis=-1)

## Loading and per-channel feature extraction

In [2]:
def find_epn_user_files(root: Path, splits=("trainingJSON",), n=None):
    files = []
    for split in splits:
        files += sorted(
            root.rglob(f"{split}/user*/user*.json"),
            key=lambda p: int("".join(filter(str.isdigit, p.stem))),
        )
    return files if n is None else files[:n]


def samples_for_gesture(user_json: dict, gesture: str, n=None):
    out = []
    for split in ("trainingSamples", "testingSamples"):
        for sample in user_json.get(split, {}).values():
            if sample.get("gestureName") == gesture:
                out.append(sample["emg"])
                if n is not None and len(out) == n:
                    return out
    return out


DEMOGRAPHIC_COLS = [
    "age",
    "is_female",
    "distanceFromElbowToMyoInCm",
    "distanceFromElbowToUlnaInCm",
    "armPerimeterInCm",
    "recording_date",
]


def user_demographics(user_json: dict) -> dict:
    ui = user_json["userInfo"]
    return {
        "age": ui["age"],
        "gender": ui["gender"],
        "is_female": float(str(ui["gender"]).strip().lower() in ("woman", "female", "w", "f")),
        "distanceFromElbowToMyoInCm": ui["distanceFromElbowToMyoInCm"],
        "distanceFromElbowToUlnaInCm": ui["distanceFromElbowToUlnaInCm"],
        "armPerimeterInCm": ui["armPerimeterInCm"],
        "recording_date": pd.to_datetime(ui["date"], format="%d-%b-%Y %H:%M:%S"),
    }


def per_channel_features(x: np.ndarray) -> dict:
    """MAV + Fourier-domain features (Welch PSD). Expects x already high-pass filtered."""
    feats = {}
    feats["mav"] = np.mean(np.abs(x))
    nperseg = min(len(x), int(FS * FOURIER_SEG_SECONDS))
    freqs, psd = welch(x, fs=FS, nperseg=nperseg)
    peak_idx = np.argmax(psd)
    feats["fft_peak_freq"] = freqs[peak_idx]
    feats["fft_mean_freq"] = np.sum(freqs * psd) / psd.sum()
    return feats


def load_wide_windows(splits=("trainingJSON",), n_users=None, trials_per_gesture=None):
    rows = []
    files = find_epn_user_files(EPN_DIR, splits=splits, n=n_users)
    multi_split = len(splits) > 1
    for k, path in enumerate(files, 1):
        user_json = json.loads(path.read_text())
        demo = user_demographics(user_json)
        uid = f"{path.parents[1].name}:{path.parent.name}" if multi_split else path.parent.name
        for gesture in GESTURES:
            for emg_dict in samples_for_gesture(user_json, gesture, trials_per_gesture):
                row = {"user": uid, "gesture": gesture, **demo}
                for ch in range(1, N_CHANNELS + 1):
                    x = apply_highpass(np.array(emg_dict[f"ch{ch}"], dtype=float))
                    for name, value in per_channel_features(x).items():
                        row[f"ch{ch}_{name}"] = value
                rows.append(row)
        if k % 25 == 0 or k == len(files):
            print(f"  loaded {k}/{len(files)} users, {len(rows)} windows so far")
    return pd.DataFrame(rows)


SPLITS = ("trainingJSON",)
N_USERS = None
TRIALS_PER_GESTURE = 3

wide_df = load_wide_windows(SPLITS, N_USERS, TRIALS_PER_GESTURE)
FEATURE_COLS_WIDE = [c for c in wide_df.columns if c.startswith("ch") and ("mav" in c or "fft" in c)]
print(f"\n{wide_df.shape[0]} rows, {len(FEATURE_COLS_WIDE)} EMG features, "
      f"age range {wide_df['age'].min()}-{wide_df['age'].max()}")
wide_df.head()

  loaded 25/306 users, 450 windows so far


  loaded 50/306 users, 900 windows so far


  loaded 75/306 users, 1350 windows so far


  loaded 100/306 users, 1800 windows so far


  loaded 125/306 users, 2250 windows so far


...

  loaded 250/306 users, 4500 windows so far


  loaded 275/306 users, 4950 windows so far


  loaded 300/306 users, 5400 windows so far


  loaded 306/306 users, 5508 windows so far

5508 rows, 24 EMG features, age range 18-54


,user,gesture,age,gender,is_female,distanceFromElbowToMyoInCm,distanceFromElbowToUlnaInCm,armPerimeterInCm,recording_date,ch1_mav,...,ch5_fft_mean_freq,ch6_mav,ch6_fft_peak_freq,ch6_fft_mean_freq,ch7_mav,ch7_fft_peak_freq,ch7_fft_mean_freq,ch8_mav,ch8_fft_peak_freq,ch8_fft_mean_freq
0,user1,noGesture,19,man,0.0,6.0,24.0,23.0,2019-10-30 14:34:28,0.895392,...,62.722870,0.957900,88.0,62.268805,0.743140,91.5,60.436754,0.764259,88.5,60.697561
1,user1,noGesture,19,man,0.0,6.0,24.0,23.0,2019-10-30 14:34:28,0.936866,...,64.008650,0.968333,73.0,60.045452,0.791993,88.0,61.995388,0.768542,82.0,61.913615
2,user1,noGesture,19,man,0.0,6.0,24.0,23.0,2019-10-30 14:34:28,1.100648,...,60.165651,0.998615,29.5,62.335222,0.795677,76.5,59.992599,0.844713,90.0,62.737565
3,user1,open,19,man,0.0,6.0,24.0,23.0,2019-10-30 14:34:28,2.536149,...,62.790041,1.933941,65.5,61.215896,1.967835,66.5,60.764028,3.952365,68.5,67.371262
4,user1,open,19,man,0.0,6.0,24.0,23.0,2019-10-30 14:34:28,2.110451,...,64.941584,1.947802,87.0,69.948563,1.773373,83.0,69.633400,2.652607,67.0,60.070690


## Causal discovery

In [3]:
import os
import subprocess
import jpype
import importlib.resources as importlib_resources

_JDK21_LIBJVM = "/home/vernon/.local/jdk/jdk-21.0.12.1+1/lib/server/libjvm.so"
if not jpype.isJVMStarted():
    _jar_path = str(importlib_resources.files("pytetrad").joinpath("resources", "tetrad-current.jar"))
    jpype.startJVM(_JDK21_LIBJVM, "-ea", "--enable-native-access=ALL-UNNAMED", classpath=[_jar_path])
print("JVM started:", jpype.getJVMVersion())

import pytetrad.tools.TetradSearch as search


GFCI_N_ROWS = 1000
GFCI_DEPTH = 2
GFCI_ALPHA = 0.05
GFCI_SEED = 0
FEATURE_COLS_ICP_TOP_N = 9


def stratified_binary_sample(gesture: str, n_rows: int = GFCI_N_ROWS, seed: int = GFCI_SEED) -> pd.DataFrame:
    """n_rows/2 positive (`gesture`) + n_rows/2 negative (every other
    gesture), encoded as strings so Tetrad treats is_gesture as
    discrete/categorical rather than continuous."""
    df = wide_df[FEATURE_COLS_WIDE].astype("float64").copy()
    is_g = wide_df["gesture"] == gesture
    df["is_gesture"] = is_g.astype(str)
    per_class = n_rows // 2
    pos = df[is_g.values].sample(n=min(per_class, is_g.sum()), random_state=seed)
    neg = df[~is_g.values].sample(n=min(per_class, (~is_g).sum()), random_state=seed)
    return pd.concat([pos, neg]).sample(frac=1, random_state=seed).reset_index(drop=True)


def run_gfci_gesture(gesture: str):
    """GFCI + Conditional Gaussian score/test on FEATURE_COLS_WIDE + is_gesture."""
    df = stratified_binary_sample(gesture)
    ts = search.TetradSearch(df)
    ts.set_verbose(False)
    ts.use_conditional_gaussian_score(penalty_discount=1)
    ts.use_conditional_gaussian_test(alpha=GFCI_ALPHA)
    ts.run_gfci(depth=GFCI_DEPTH)
    return ts, list(df.columns)


def classify_edge(ts, cols, feature: str, label: str = "is_gesture"):
    """Read the PAG edge between `feature` and `label` off Tetrad's
    endpoint-coded adjacency matrix (0=none, 1=circle, 2=arrow, 3=tail).
    `mat[i, j]` = the mark AT j's end of the edge between i and j."""
    mat = ts.get_graph_to_matrix().values
    i, j = cols.index(feature), cols.index(label)
    mark_at_label, mark_at_feature = mat[i, j], mat[j, i]
    if mark_at_label == 0 and mark_at_feature == 0:
        return None  # no edge at all
    if mark_at_label == 2 and mark_at_feature == 3:
        return "feature --> gesture"
    if mark_at_label == 3 and mark_at_feature == 2:
        return "gesture --> feature"
    if mark_at_label == 2 and mark_at_feature == 2:
        return "confounded (<->)"
    if mark_at_label == 2 and mark_at_feature == 1:
        return "feature o-> gesture (gesture ruled out as cause)"
    if mark_at_label == 1 and mark_at_feature == 2:
        return "gesture o-> feature (feature ruled out as cause)"
    return f"undetermined (o-o / mark_at_label={mark_at_label}, mark_at_feature={mark_at_feature})"


def edge_strength(ts, cols, feature: str, label: str = "is_gesture"):
    """BIC score contribution of a direct edge between `feature` and `label`
    (empty conditioning set): localScoreDiff(feature, label, []), read off
    the real ConditionalGaussianScore object underlying GFCI's own search --
    ts.SCORE is only a lightweight ScoreWrapper factory, so the actual
    scorer has to be built via ts.SCORE.getScore(ts.get_data(), ts.params)
    first. This is the exact same BIC score/penalty math GFCI itself used to
    decide whether to keep this edge -- larger = more strongly justified;
    negative = the BIC penalty outweighs the fit improvement (the score
    would rather not have the edge at all)."""
    real_score = ts.SCORE.getScore(ts.get_data(), ts.params)
    feature_idx, label_idx = cols.index(feature), cols.index(label)
    empty_z = jpype.JArray(jpype.JInt)([])
    return float(real_score.localScoreDiff(feature_idx, label_idx, empty_z))


GFCI_PLOT_DIR = Path("gfci_plots")
GFCI_PLOT_DIR.mkdir(exist_ok=True)

_GRAPHVIZ_BIN = Path.home() / ".local/graphviz/extracted/usr/bin/dot"
_GRAPHVIZ_LIB = str(Path.home() / ".local/graphviz/extracted/usr/lib/x86_64-linux-gnu")
_GRAPHVIZ_LIB_PLUGIN = _GRAPHVIZ_LIB + "/graphviz"


def save_pag_plot(ts, gesture: str, plot_dir: Path) -> Path:
    """Render the PAG to PNG via TetradSearch.get_dot() piped through dot."""
    dot_src = ts.get_dot()
    out_path = plot_dir / f"gfci_pag_{gesture}.png"
    env = {**os.environ, "LD_LIBRARY_PATH": f"{_GRAPHVIZ_LIB}:{_GRAPHVIZ_LIB_PLUGIN}"}
    subprocess.run([str(_GRAPHVIZ_BIN), "-Tpng", "-o", str(out_path)],
                    input=dot_src, text=True, env=env, check=True)
    return out_path


ACTIVE_GESTURES = [g for g in GESTURES if g != "noGesture"]

gfci_edges = {}
edge_strengths = {}  # {gesture: {feature: BIC score contribution}}
for gesture in GESTURES:
    ts, cols = run_gfci_gesture(gesture)
    edges = {f: classify_edge(ts, cols, f) for f in FEATURE_COLS_WIDE}
    edges = {f: kind for f, kind in edges.items() if kind is not None}
    gfci_edges[gesture] = edges
    if gesture in ACTIVE_GESTURES:
        edge_strengths[gesture] = {f: edge_strength(ts, cols, f) for f in FEATURE_COLS_WIDE}
    plot_path = save_pag_plot(ts, gesture, GFCI_PLOT_DIR)
    print(f"{gesture}: {len(edges)} feature(s) connected to is_gesture -> saved {plot_path}", flush=True)
    for f, kind in edges.items():
        print(f"    {f}: {kind}")

# Rank FEATURE_COLS_WIDE by total GFCI edge strength (BIC score contribution,
# summed across the 5 active gestures) and keep only the top N. This replaces
# the earlier "union of any-edge features" selection -- which grew to 17
# features and made ICP's full 2**p subset search intractable (131,072
# subsets) -- with a fixed-size, strength-ranked shortlist small enough for
# ICP to search exhaustively again (2**9 = 512, matching the original scale).
total_strength = {
    f: sum(edge_strengths[g][f] for g in ACTIVE_GESTURES)
    for f in FEATURE_COLS_WIDE
}
ranked_features = sorted(total_strength, key=total_strength.get, reverse=True)
FEATURE_COLS_ICP = ranked_features[:FEATURE_COLS_ICP_TOP_N]

print(f"\nFeatures ranked by total GFCI edge strength (summed BIC score contribution "
      f"across the 5 active gestures), top {FEATURE_COLS_ICP_TOP_N} selected:")
for f in ranked_features:
    marker = " <-- selected" if f in FEATURE_COLS_ICP else ""
    print(f"    {f}: {total_strength[f]:.2f}{marker}")

print(f"\n{len(FEATURE_COLS_ICP)} candidate features for ICP "
      f"({2**len(FEATURE_COLS_ICP)} subsets to test per gesture):")
print(FEATURE_COLS_ICP)

JVM started: (21, 0, 12, 1)


Sep 15, 2026 10:47:09 PM java.util.prefs.FileSystemPreferences$6 run


noGesture: 23 feature(s) connected to is_gesture -> saved gfci_plots/gfci_pag_noGesture.png


    ch1_mav: gesture --> feature
    ch1_fft_peak_freq: gesture --> feature
    ch1_fft_mean_freq: gesture --> feature
    ch2_mav: gesture --> feature
    ch2_fft_peak_freq: gesture --> feature
    ch2_fft_mean_freq: gesture --> feature
    ch3_mav: undetermined (o-o / mark_at_label=1, mark_at_feature=1)
    ch3_fft_peak_freq: gesture --> feature
    ch3_fft_mean_freq: gesture --> feature
    ch4_mav: undetermined (o-o / mark_at_label=1, mark_at_feature=1)
    ch4_fft_mean_freq: undetermined (o-o / mark_at_label=1, mark_at_feature=1)
    ch5_mav: undetermined (o-o / mark_at_label=1, mark_at_feature=1)
    ch5_fft_peak_freq: gesture --> feature
    ch5_fft_mean_freq: gesture --> feature
    ch6_mav: gesture --> feature
    ch6_fft_peak_freq: gesture --> feature
    ch6_fft_mean_freq: gesture --> feature
    ch7_mav: gesture --> feature
    ch7_fft_peak_freq: gesture --> feature
    ch7_fft_mean_freq: gesture --> feature
    ch8_mav: gesture --> feature
    ch8_fft_peak_freq: gesture --

open: 7 feature(s) connected to is_gesture -> saved gfci_plots/gfci_pag_open.png


...

    ch1_fft_peak_freq: gesture --> feature
    ch1_fft_mean_freq: gesture --> feature
    ch3_mav: gesture --> feature
    ch3_fft_mean_freq: gesture --> feature
    ch4_mav: feature --> gesture
    ch4_fft_mean_freq: gesture --> feature
    ch5_fft_peak_freq: gesture --> feature
    ch6_fft_mean_freq: gesture --> feature
    ch8_mav: feature --> gesture
    ch8_fft_mean_freq: gesture --> feature


fist: 8 feature(s) connected to is_gesture -> saved gfci_plots/gfci_pag_fist.png


    ch1_mav: gesture o-> feature (feature ruled out as cause)
    ch1_fft_peak_freq: gesture --> feature
    ch2_mav: undetermined (o-o / mark_at_label=1, mark_at_feature=1)
    ch3_fft_mean_freq: gesture --> feature
    ch4_mav: gesture --> feature
    ch4_fft_mean_freq: gesture --> feature
    ch5_mav: gesture --> feature
    ch8_mav: gesture --> feature


waveIn: 9 feature(s) connected to is_gesture -> saved gfci_plots/gfci_pag_waveIn.png


    ch2_mav: gesture --> feature
    ch4_mav: gesture --> feature
    ch5_mav: undetermined (o-o / mark_at_label=1, mark_at_feature=1)
    ch5_fft_mean_freq: gesture --> feature
    ch6_mav: undetermined (o-o / mark_at_label=1, mark_at_feature=1)
    ch6_fft_mean_freq: gesture --> feature
    ch7_mav: gesture --> feature
    ch7_fft_mean_freq: gesture --> feature
    ch8_mav: undetermined (o-o / mark_at_label=1, mark_at_feature=1)

Features ranked by total GFCI edge strength (summed BIC score contribution across the 5 active gestures), top 9 selected:
    ch1_mav: 495.10 <-- selected
    ch8_mav: 414.27 <-- selected
    ch7_mav: 339.75 <-- selected
    ch4_mav: 299.51 <-- selected
    ch2_mav: 283.69 <-- selected
    ch5_mav: 208.75 <-- selected
    ch5_fft_mean_freq: 174.51 <-- selected
    ch6_mav: 161.93 <-- selected
    ch3_mav: 155.71 <-- selected
    ch1_fft_mean_freq: 96.62
    ch4_fft_mean_freq: 65.40
    ch2_fft_mean_freq: 58.55
    ch3_fft_mean_freq: 46.95
    ch8_fft_mean_fr

In [4]:
import gc

if jpype.isJVMStarted():
    jpype.shutdownJVM()
gc.collect()

0

## Save outputs for the downstream age/distance ICP notebooks

`age_breakup/EMG_causal_modeling.ipynb` and
`electrode_placement/EMG_causal_modeling.ipynb` load these instead of
re-running data loading + GFCI themselves.

In [5]:
import pickle

with open(OUTPUT_DIR / "wide_df.pkl", "wb") as f:
    pickle.dump(wide_df, f)

with open(OUTPUT_DIR / "feature_cols_icp.json", "w") as f:
    json.dump(FEATURE_COLS_ICP, f, indent=2)

with open(OUTPUT_DIR / "feature_cols_wide.json", "w") as f:
    json.dump(FEATURE_COLS_WIDE, f, indent=2)

with open(OUTPUT_DIR / "gfci_edges.json", "w") as f:
    json.dump(gfci_edges, f, indent=2)

with open(OUTPUT_DIR / "feature_edge_strength.json", "w") as f:
    json.dump({"per_gesture": edge_strengths, "total": total_strength}, f, indent=2)

print("Saved to", OUTPUT_DIR.resolve())
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" ", p.name)

Saved to /home/vernon/EMG-causal-analysis/notebooks/causal_discovery/output
  feature_cols_icp.json
  feature_cols_wide.json
  feature_edge_strength.json
  gfci_edges.json
  wide_df.pkl
